# A small LowLevelFEM example: high-level API vs. executable weak form

This notebook solves the same simple 3D linear-elasticity problem in two different ways.

1. First, the problem is solved through LowLevelFEM's traditional high-level elasticity interface.
2. Then the same problem is written at the field/operator level, where the weak form is expressed directly in Julia code.

The point is not to present a serious benchmark suite, but to show how close the lower-level formulation can stay to the mathematics while still assembling and solving the finite element system directly.


## Mesh, material, and benchmark setup

A structured 3D box mesh is generated with Gmsh through LowLevelFEM, and a linear elastic material is assigned to the physical group `"body"`.

`BenchmarkTools.@btime` is used for the timings below. Variables created outside a benchmark expression are interpolated with `$` so that access to globals does not distort the measurement.


In [1]:
using LowLevelFEM, LinearAlgebra
using BenchmarkTools

structured_box_mesh(n=10)
mat = Material("body", E=2e5, ν=0.3);

## 1. High-level elasticity interface

This is the compact, problem-specific interface. `Problem([mat], type=:Solid)` creates a standard elasticity problem.

Inside the benchmark we define a fixed support on the `"left"` boundary and a unit force in the \(x\)-direction on the `"right"` boundary. `solveDisplacement` then performs the finite-element assembly, applies the boundary conditions, and solves for the displacement field.

This interface is convenient when the governing equations are already built into LowLevelFEM.


In [2]:
P1 = Problem([mat], type=:Solid)

u = @btime begin
    bc = displacementConstraint("left", ux=0, uy=0, uz=0)
    ld = load("right", fx=1)
    solveDisplacement($P1, load=[ld], support=[bc])
end;

  305.827 ms (577669 allocations: 298.94 MiB)


### Stress recovery with the high-level interface

The displacement field returned above is passed to `solveStress`, which evaluates the corresponding stress field using the built-in elasticity formulation.


In [3]:
@btime σ = solveStress($u);

  31.611 ms (628241 allocations: 47.07 MiB)


## 2. The same problem from the weak form

Here the displacement is introduced explicitly as a vector field, `P2`. The isotropic elasticity matrix \(D\) is constructed from the Lamé parameters.

The stiffness matrix is assembled from

$$
K = \int_\Omega \varepsilon(v)^\mathsf{T} D\,\varepsilon(u)\,d\Omega ,
$$

which appears in the code as

```julia
∫(SymGrad(P2) ⋅ D ⋅ SymGrad(P2))
```

The external load is another integral,

$$
f = \int_\Gamma v \cdot t\,d\Gamma ,
$$

and is written directly as

```julia
∫(P2 ⋅ [1.0, 0.0, 0.0], Γ="right")
```

The assembled matrix and vector are then passed to the generic `solveField` routine. In other words, this version exposes the mathematical operators and the weak form instead of calling a problem-specific stiffness-matrix routine.


In [4]:
P2 = Problem([mat], type=:VectorField, dim=3, field=:u)

u = @btime begin
    μ = $mat.μ
    λ = $mat.λ
    D = [λ+2μ λ λ 0 0 0; λ λ+2μ λ 0 0 0; λ λ λ+2μ 0 0 0; 0 0 0 μ 0 0; 0 0 0 0 μ 0; 0 0 0 0 0 μ]

    K = ∫(SymGrad($P2) ⋅ D ⋅ SymGrad($P2))

    f = ∫($P2 ⋅ [1.0, 0.0, 0.0], Γ="right")

    bc = BoundaryCondition("left", ux=0, uy=0, uz=0)

    solveField(Symmetric(K), f, support=[bc])
end;

  120.004 ms (13150 allocations: 52.21 MiB)


### Stress as field algebra

Post-processing can be written in the same operator language.

The first line constructs the infinitesimal strain tensor,

$$
\varepsilon = \frac{1}{2}\left(\nabla u + (\nabla u)^\mathsf{T}\right),
$$

and the final expression evaluates the isotropic Hooke law

$$
\sigma =
\frac{E}{1+\nu}
\left(
\varepsilon +
\frac{\nu}{1-2\nu}\,\mathrm{tr}(\varepsilon)\,I
\right).
$$

So the stress calculation below is not a dedicated `solveStress` call: it is ordinary algebra on scalar, vector, and tensor fields.


In [5]:
σ = @btime begin
    ε = ($u ∘ ∇ + ∇ ∘ $u) / 2.0

    I = TensorField($P2, "body", [1 0 0; 0 1 0; 0 0 1])
    E = $mat.E
    ν = $mat.ν

    E / (1 + ν) * (ε + ν / (1 - 2ν) * trace(ε) * I)
end;


  26.006 ms (117187 allocations: 13.59 MiB)


## What this example is meant to show

The two sections solve the same physical problem, but at different abstraction levels. The first is shorter and more specialized; the second exposes the weak form and constitutive equations directly.

The lower-level form is useful for experimenting with coupled or non-standard PDEs because the operators, integrals, fields, and algebra can be combined without adding a new problem-specific solver for every equation.

The timings are included only as a practical reference for this particular mesh and machine. They should not be interpreted as a general performance comparison without a more systematic benchmark.


In [6]:
showDoFResults(u, name="u", visible=true)
showElementResults(σ, name="σ")
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
